In [1]:
# Cell 1: tools / engine
# ------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, HTML, Markdown, clear_output

# ---------- basic single-qubit objects ----------
I2 = np.eye(2, dtype=complex)
Y  = np.array([[0, -1j], [1j, 0]], dtype=complex)

zero = np.array([1, 0], dtype=complex)
one  = np.array([0, 1], dtype=complex)

def normalize(v):
    nrm = np.linalg.norm(v)
    return v if nrm == 0 else v / nrm

def dm(ket):
    ket = ket.reshape(-1, 1)
    return ket @ ket.conj().T

def kron_all(*ops):
    out = np.array([1.0 + 0.0j])
    for op in ops:
        out = np.kron(out, op)
    return out

# ---------- two-qubit basis states ----------
ket00 = np.kron(zero, zero)
ket01 = np.kron(zero, one)
ket10 = np.kron(one,  zero)
ket11 = np.kron(one,  one)

# ---------- source states ----------
def source_state(alpha_deg=45.0, phase_deg=0.0):
    """
    |psi(alpha,phi)> = cos(alpha)|00> + exp(i phi) sin(alpha)|11>
    """
    a  = np.deg2rad(alpha_deg)
    ph = np.deg2rad(phase_deg)
    ket = np.cos(a) * ket00 + np.exp(1j * ph) * np.sin(a) * ket11
    return normalize(ket)

# ---------- Bell states on 2 qubits ----------
BELL_KETS = {
    "Phi+": normalize(ket00 + ket11),
    "Phi-": normalize(ket00 - ket11),
    "Psi+": normalize(ket01 + ket10),
    "Psi-": normalize(ket01 - ket10),
}
BELL_DMS = {k: dm(v) for k, v in BELL_KETS.items()}

# ---------- rotated / misaligned BSM basis ----------
def rotated_bsm_basis(gamma_even_deg=45.0, phi_even_deg=0.0,
                      gamma_odd_deg=45.0,  phi_odd_deg=0.0):
    """
    Hub Bell measurement basis with independent rotations in the even and odd sectors.

    even sector:
        |B_e1> = cos(g)|00> + exp(i phi_e) sin(g)|11>
        |B_e2> = sin(g)|00> - exp(i phi_e) cos(g)|11>

    odd sector:
        |B_o1> = cos(d)|01> + exp(i phi_o) sin(d)|10>
        |B_o2> = sin(d)|01> - exp(i phi_o) cos(d)|10>

    Bell basis recovered at:
        gamma_even = 45 deg, phi_even = 0
        gamma_odd  = 45 deg, phi_odd  = 0
    """
    g  = np.deg2rad(gamma_even_deg)
    pe = np.deg2rad(phi_even_deg)

    d  = np.deg2rad(gamma_odd_deg)
    po = np.deg2rad(phi_odd_deg)

    B_e1 = normalize(np.cos(g) * ket00 + np.exp(1j * pe) * np.sin(g) * ket11)
    B_e2 = normalize(np.sin(g) * ket00 - np.exp(1j * pe) * np.cos(g) * ket11)

    B_o1 = normalize(np.cos(d) * ket01 + np.exp(1j * po) * np.sin(d) * ket10)
    B_o2 = normalize(np.sin(d) * ket01 - np.exp(1j * po) * np.cos(d) * ket10)

    return {
        "B_e1": B_e1,
        "B_e2": B_e2,
        "B_o1": B_o1,
        "B_o2": B_o2,
    }

# ---------- partial trace ----------
def partial_trace(rho, keep, dims):
    """
    Partial trace over all subsystems not in 'keep'.
    rho  : full density matrix
    keep : list of subsystem indices to keep, 0-based
    dims : list of subsystem dimensions
    """
    keep = list(keep)
    n = len(dims)
    trace_out = [i for i in range(n) if i not in keep]

    reshaped = rho.reshape(*(dims + dims))
    current_n = n

    for ax in sorted(trace_out, reverse=True):
        reshaped = np.trace(reshaped, axis1=ax, axis2=ax + current_n)
        current_n -= 1

    out_dim = int(np.prod([dims[i] for i in keep]))
    return reshaped.reshape(out_dim, out_dim)

# ---------- BSM projection on qubits 2 and 3 ----------
def conditional_remote_state(rho1234, bell_vec_23):
    """
    Four-qubit ordering is (1,2,3,4).
    Measure qubits 2 and 3 in state |bell_vec_23>.
    Return:
        p_branch, rho14_conditional
    """
    proj23 = np.outer(bell_vec_23, bell_vec_23.conj())  # 4x4
    P_full = np.kron(np.kron(I2, proj23), I2)           # 16x16

    post = P_full @ rho1234 @ P_full
    p = float(np.real_if_close(np.trace(post)))

    rho14 = partial_trace(post, keep=[0, 3], dims=[2, 2, 2, 2])
    if p > 1e-15:
        rho14 = rho14 / p

    return p, rho14

# ---------- entanglement metrics ----------
def concurrence(rho):
    """
    Wootters concurrence for a 2-qubit density matrix.
    """
    YY = np.kron(Y, Y)
    R = rho @ YY @ rho.conj() @ YY

    eigvals = np.linalg.eigvals(R)
    eigvals = np.real_if_close(eigvals)
    eigvals = np.clip(np.real(eigvals), 0.0, None)
    s = np.sort(np.sqrt(eigvals))[::-1]

    return float(max(0.0, s[0] - s[1] - s[2] - s[3]))

def partial_transpose_2q(rho, sys=1):
    rr = rho.reshape(2, 2, 2, 2)
    if sys == 0:
        rr = rr.transpose(2, 1, 0, 3)
    else:
        rr = rr.transpose(0, 3, 2, 1)
    return rr.reshape(4, 4)

def negativity(rho):
    """
    Sum of absolute values of negative eigenvalues of the partial transpose.
    """
    pt = partial_transpose_2q(rho, sys=1)
    evals = np.linalg.eigvalsh((pt + pt.conj().T) / 2)
    return float(np.sum(np.abs(evals[evals < 0])))

def bell_fidelities(rho):
    vals = {}
    for label, bell_dm in BELL_DMS.items():
        vals[label] = float(np.real(np.trace(rho @ bell_dm)))
    best_label = max(vals, key=vals.get)
    best_val = max(0.0, min(1.0, vals[best_label]))
    return vals, best_label, best_val

# ---------- main simulation ----------
def simulate_swap(alpha_A_deg=45.0, phase_A_deg=0.0,
                  alpha_B_deg=45.0, phase_B_deg=0.0,
                  gamma_even_deg=45.0, phi_even_deg=0.0,
                  gamma_odd_deg=45.0,  phi_odd_deg=0.0,
                  eta2=1.0, eta3=1.0, xi_bsm=1.0):
    """
    Simple network-facing model:
      - state quality determined by source states + BSM alignment
      - valid heralded event rate reduced by eta2 * eta3 * xi_bsm

    eta2, eta3 : transmission from sources to hub on legs 2 and 3
    xi_bsm     : valid BSM efficiency factor
    """
    psi12 = source_state(alpha_A_deg, phase_A_deg)
    psi34 = source_state(alpha_B_deg, phase_B_deg)

    psi1234 = np.kron(psi12, psi34)
    rho1234 = dm(psi1234)

    basis = rotated_bsm_basis(
        gamma_even_deg=gamma_even_deg,
        phi_even_deg=phi_even_deg,
        gamma_odd_deg=gamma_odd_deg,
        phi_odd_deg=phi_odd_deg,
    )

    link_factor = float(np.clip(eta2, 0.0, 1.0) * np.clip(eta3, 0.0, 1.0) * np.clip(xi_bsm, 0.0, 1.0))

    results = {}
    total_conditional_prob = 0.0
    total_valid_prob = 0.0
    total_ent_yield = 0.0
    total_fid_yield = 0.0

    for label, bvec in basis.items():
        p_cond, rho14 = conditional_remote_state(rho1234, bvec)
        p_valid = link_factor * p_cond

        C = concurrence(rho14) if p_cond > 1e-15 else 0.0
        N = negativity(rho14) if p_cond > 1e-15 else 0.0
        bfids, best_bell, best_fid = bell_fidelities(rho14) if p_cond > 1e-15 else ({}, "-", 0.0)

        ent_yield = p_valid * C
        fid_yield = p_valid * best_fid

        results[label] = {
            "p_cond": p_cond,
            "p_valid": p_valid,
            "concurrence": C,
            "negativity": N,
            "best_bell": best_bell,
            "best_bell_fidelity": best_fid,
            "ent_yield": ent_yield,
            "fid_yield": fid_yield,
            "rho14": rho14,
            "bell_fidelities": bfids,
        }

        total_conditional_prob += p_cond
        total_valid_prob += p_valid
        total_ent_yield += ent_yield
        total_fid_yield += fid_yield

    summary = {
        "link_factor": link_factor,
        "total_conditional_prob": total_conditional_prob,
        "total_valid_prob": total_valid_prob,
        "total_ent_yield": total_ent_yield,
        "total_fid_yield": total_fid_yield,
    }

    return {"branches": results, "summary": summary}

# ---------- tabular helper ----------
def results_dataframe(sim):
    rows = []
    for label, d in sim["branches"].items():
        rows.append({
            "branch": label,
            "p_cond": d["p_cond"],
            "p_valid": d["p_valid"],
            "concurrence": d["concurrence"],
            "negativity": d["negativity"],
            "best_bell": d["best_bell"],
            "best_bell_fidelity": d["best_bell_fidelity"],
            "ent_yield": d["ent_yield"],
            "fid_yield": d["fid_yield"],
        })
    df = pd.DataFrame(rows)
    return df.sort_values("branch").reset_index(drop=True)

# ---------- heatmap helper ----------
def extract_metric(sim, branch, metric):
    if branch == "TOTAL":
        if metric == "p_cond":
            return sim["summary"]["total_conditional_prob"]
        elif metric == "p_valid":
            return sim["summary"]["total_valid_prob"]
        elif metric == "ent_yield":
            return sim["summary"]["total_ent_yield"]
        elif metric == "fid_yield":
            return sim["summary"]["total_fid_yield"]
        elif metric == "mean_concurrence":
            vals = [d["concurrence"] for d in sim["branches"].values()]
            return float(np.mean(vals))
        elif metric == "mean_fidelity":
            vals = [d["best_bell_fidelity"] for d in sim["branches"].values()]
            return float(np.mean(vals))
        else:
            raise ValueError(f"Unsupported TOTAL metric: {metric}")
    else:
        d = sim["branches"][branch]
        if metric not in d:
            raise ValueError(f"Unsupported branch metric: {metric}")
        return d[metric]

def alpha_beta_grid(params, branch="B_e1", metric="concurrence",
                    n_alpha=81, n_beta=81):
    alphas = np.linspace(0.0, 90.0, n_alpha)
    betas  = np.linspace(0.0, 90.0, n_beta)

    Z = np.zeros((n_beta, n_alpha), dtype=float)
    for i, beta in enumerate(betas):
        for j, alpha in enumerate(alphas):
            sim = simulate_swap(
                alpha_A_deg=alpha,
                phase_A_deg=params["phase_A_deg"],
                alpha_B_deg=beta,
                phase_B_deg=params["phase_B_deg"],
                gamma_even_deg=params["gamma_even_deg"],
                phi_even_deg=params["phi_even_deg"],
                gamma_odd_deg=params["gamma_odd_deg"],
                phi_odd_deg=params["phi_odd_deg"],
                eta2=params["eta2"],
                eta3=params["eta3"],
                xi_bsm=params["xi_bsm"],
            )
            Z[i, j] = extract_metric(sim, branch, metric)

    return alphas, betas, Z

# ---------- pretty print ----------
def matrix_to_html(rho, precision=3):
    arr = np.array(rho)
    rows = []
    for r in arr:
        cells = []
        for z in r:
            zr = np.real(z)
            zi = np.imag(z)
            if abs(zi) < 10**(-precision):
                txt = f"{zr:.{precision}f}"
            elif abs(zr) < 10**(-precision):
                txt = f"{zi:.{precision}f}j"
            else:
                sign = "+" if zi >= 0 else "-"
                txt = f"{zr:.{precision}f}{sign}{abs(zi):.{precision}f}j"
            cells.append(f"<td style='padding:4px 8px; text-align:right;'>{txt}</td>")
        rows.append("<tr>" + "".join(cells) + "</tr>")
    return "<table style='border-collapse:collapse;'>" + "".join(rows) + "</table>"

# ---------- defaults ----------
DEFAULT_PARAMS = {
    "alpha_A_deg": 45.0,
    "phase_A_deg": 0.0,
    "alpha_B_deg": 45.0,
    "phase_B_deg": 0.0,
    "gamma_even_deg": 45.0,
    "phi_even_deg": 0.0,
    "gamma_odd_deg": 45.0,
    "phi_odd_deg": 0.0,
    "eta2": 1.0,
    "eta3": 1.0,
    "xi_bsm": 1.0,
}

# ---------- shared UI/display labels ----------
UI_LABELS = {
    "alpha_A_deg": "α_A (deg)",
    "phase_A_deg": "φ_A (deg)",
    "alpha_B_deg": "α_B (deg)",
    "phase_B_deg": "φ_B (deg)",
    "gamma_even_deg": "γ_even (deg)",
    "phi_even_deg": "φ_even (deg)",
    "gamma_odd_deg": "γ_odd (deg)",
    "phi_odd_deg": "φ_odd (deg)",
    "eta2": "η₂",
    "eta3": "η₃",
    "xi_bsm": "ξ_BSM",
}

def build_parameter_lines(params, extra=None):
    """
    Build a list of formatted parameter lines using the exact same labels
    as the UI controls.
    """
    ordered_keys = [
        "alpha_A_deg",
        "phase_A_deg",
        "alpha_B_deg",
        "phase_B_deg",
        "gamma_even_deg",
        "phi_even_deg",
        "gamma_odd_deg",
        "phi_odd_deg",
        "eta2",
        "eta3",
        "xi_bsm",
    ]

    lines = []
    for k in ordered_keys:
        label = UI_LABELS[k]
        v = params[k]
        if "deg" in k:
            lines.append(f"{label} = {v:.1f}")
        else:
            lines.append(f"{label} = {v:.2f}")

    if extra:
        for label, value in extra:
            lines.append(f"{label} = {value}")

    return lines
    
# global cache shared by Cell 2 / Cell 3
SWAP_APP = {
    "params": DEFAULT_PARAMS.copy(),
    "sim": simulate_swap(**DEFAULT_PARAMS),
}

In [10]:
# Cell 2: UI
# ------------------------------------------------------------

# ---------- controls ----------
alpha_A = widgets.FloatSlider(
    description="α_A (deg)", min=0, max=90, step=1,
    value=SWAP_APP["params"]["alpha_A_deg"], continuous_update=False
)
phase_A = widgets.FloatSlider(
    description="φ_A (deg)", min=-180, max=180, step=5,
    value=SWAP_APP["params"]["phase_A_deg"], continuous_update=False
)

alpha_B = widgets.FloatSlider(
    description="α_B (deg)", min=0, max=90, step=1,
    value=SWAP_APP["params"]["alpha_B_deg"], continuous_update=False
)
phase_B = widgets.FloatSlider(
    description="φ_B (deg)", min=-180, max=180, step=5,
    value=SWAP_APP["params"]["phase_B_deg"], continuous_update=False
)

gamma_even = widgets.FloatSlider(
    description="γ_even (deg)", min=0, max=90, step=1,
    value=SWAP_APP["params"]["gamma_even_deg"], continuous_update=False
)
phi_even = widgets.FloatSlider(
    description="φ_even (deg)", min=-180, max=180, step=5,
    value=SWAP_APP["params"]["phi_even_deg"], continuous_update=False
)

gamma_odd = widgets.FloatSlider(
    description="γ_odd (deg)", min=0, max=90, step=1,
    value=SWAP_APP["params"]["gamma_odd_deg"], continuous_update=False
)
phi_odd = widgets.FloatSlider(
    description="φ_odd (deg)", min=-180, max=180, step=5,
    value=SWAP_APP["params"]["phi_odd_deg"], continuous_update=False
)

eta2 = widgets.FloatSlider(
    description="η₂", min=0.0, max=1.0, step=0.01,
    value=SWAP_APP["params"]["eta2"], continuous_update=False
)
eta3 = widgets.FloatSlider(
    description="η₃", min=0.0, max=1.0, step=0.01,
    value=SWAP_APP["params"]["eta3"], continuous_update=False
)
xi_bsm = widgets.FloatSlider(
    description="ξ_BSM", min=0.0, max=1.0, step=0.01,
    value=SWAP_APP["params"]["xi_bsm"], continuous_update=False
)

focus_branch = widgets.Dropdown(
    description="Focus branch",
    options=["B_e1", "B_e2", "B_o1", "B_o2"],
    value="B_e1",
)

run_btn = widgets.Button(description="Compute snapshot", button_style="primary")
out = widgets.Output()

def collect_params():
    return {
        "alpha_A_deg": alpha_A.value,
        "phase_A_deg": phase_A.value,
        "alpha_B_deg": alpha_B.value,
        "phase_B_deg": phase_B.value,
        "gamma_even_deg": gamma_even.value,
        "phi_even_deg": phi_even.value,
        "gamma_odd_deg": gamma_odd.value,
        "phi_odd_deg": phi_odd.value,
        "eta2": eta2.value,
        "eta3": eta3.value,
        "xi_bsm": xi_bsm.value,
    }

def update_snapshot(_=None):
    params = collect_params()
    sim = simulate_swap(**params)

    SWAP_APP["params"] = params
    SWAP_APP["sim"] = sim

    df = results_dataframe(sim)
    branch = focus_branch.value
    b = sim["branches"][branch]

    left = widgets.Output(layout=widgets.Layout(width="64%"))
    right = widgets.Output(layout=widgets.Layout(width="36%"))

    with left:
        clear_output(wait=True)

        display(Markdown("### Snapshot"))
        display(df.style.format({
            "p_cond": "{:.4f}",
            "p_valid": "{:.4f}",
            "concurrence": "{:.4f}",
            "negativity": "{:.4f}",
            "best_bell_fidelity": "{:.4f}",
            "ent_yield": "{:.4f}",
            "fid_yield": "{:.4f}",
        }))

        focused_left = widgets.HTML(
    value=f"""
<div style="line-height:1.25; margin:0; padding:0;">
  <b>Focused branch:</b> <code>{branch}</code><br>
  <b>best Bell match:</b> <code>{b['best_bell']}</code><br>
  <b>conditional probability:</b> {b['p_cond']:.4f}<br>
  <b>valid probability after loss:</b> {b['p_valid']:.4f}<br>
  <b>concurrence:</b> {b['concurrence']:.4f}
</div>
"""
)

        focused_right = widgets.HTML(
    value=f"""
<div style="line-height:1.25; margin:0; padding:0;">
  <br><b>negativity:</b> {b['negativity']:.4f}<br>
  <b>best Bell fidelity:</b> {b['best_bell_fidelity']:.4f}<br>
  <b>entanglement yield:</b> {b['ent_yield']:.4f}<br>
  <b>fidelity yield:</b> {b['fid_yield']:.4f}
</div>
"""
)

        display(
            widgets.HBox(
                [focused_left, focused_right],
                layout=widgets.Layout(
                    width="100%",
                    align_items="flex-start",
                    justify_content="space-between",
                ),
            )
        )

        display(Markdown(
            """
**Network-facing interpretation**
- `p_cond` = conditional branch probability from the source state + BSM geometry
- `p_valid` = valid branch probability after link loss and BSM efficiency
- `ent_yield = p_valid × concurrence`
- `fid_yield = p_valid × best Bell fidelity`
"""
        ))

    with right:
        clear_output(wait=True)
        display(Markdown("### Conditional remote state"))
        display(HTML(f"<b>ρ<sub>14|{branch}</sub></b>"))
        display(HTML(matrix_to_html(b["rho14"], precision=3)))

    with out:
        clear_output(wait=True)
        display(
            widgets.HBox(
                [left, right],
                layout=widgets.Layout(
                    width="100%",
                    align_items="flex-start",
                    justify_content="space-between",
                ),
            )
        )

controls_left = widgets.VBox([alpha_A, phase_A, alpha_B, phase_B])
controls_mid = widgets.VBox([gamma_even, phi_even, gamma_odd, phi_odd])
controls_right = widgets.VBox([eta2, eta3, xi_bsm, focus_branch, run_btn])

ui = widgets.HBox([controls_left, controls_mid, controls_right])

run_btn.on_click(update_snapshot)

display(Markdown("## Entanglement-swapping sandbox — controls"))
display(ui)
display(out)

# initialize once
update_snapshot()

## Entanglement-swapping sandbox — controls

Output()

In [3]:
# Cell 3: figures
# ------------------------------------------------------------

heatmap_branch = widgets.Dropdown(
    description="Heatmap branch",
    options=["B_e1", "B_e2", "B_o1", "B_o2", "TOTAL"],
    value="B_e1",
)

heatmap_metric = widgets.Dropdown(
    description="Heatmap metric",
    options=[
        "p_cond",
        "p_valid",
        "concurrence",
        "negativity",
        "best_bell_fidelity",
        "ent_yield",
        "fid_yield",
        "mean_concurrence",
        "mean_fidelity",
    ],
    value="concurrence",
)

grid_pts = widgets.IntSlider(
    description="Grid",
    min=31,
    max=121,
    step=10,
    value=81,
    continuous_update=False,
)

plot_btn = widgets.Button(description="Render figures", button_style="success")
plot_out = widgets.Output()

def render_figures(_=None):
    params = SWAP_APP["params"]
    sim = SWAP_APP["sim"]

    df = results_dataframe(sim)

    branch = heatmap_branch.value
    metric = heatmap_metric.value
    npts = grid_pts.value

    with plot_out:
        clear_output(wait=True)

        # ---------- heatmap data ----------
        alphas, betas, Z = alpha_beta_grid(
            params=params,
            branch=branch,
            metric=metric,
            n_alpha=npts,
            n_beta=npts,
        )

        # ---------- bar-chart data ----------
        branches = df["branch"].tolist()
        p_cond = df["p_cond"].values
        p_valid = df["p_valid"].values
        conc = df["concurrence"].values
        fid  = df["best_bell_fidelity"].values

        # ---------- figure ----------
        fig, axs = plt.subplots(2, 2, figsize=(13, 10))

        # (1) branch probabilities
        ax = axs[0, 0]
        x = np.arange(len(branches))
        w = 0.38
        ax.bar(x - w/2, p_cond, width=w, label="p_cond")
        ax.bar(x + w/2, p_valid, width=w, label="p_valid")
        ax.set_xticks(x)
        ax.set_xticklabels(branches)
        ax.set_ylabel("Probability")
        ax.set_title("Branch probabilities")
        ax.legend()

        # (2) branch quality
        ax = axs[0, 1]
        ax.bar(x - w/2, conc, width=w, label="concurrence")
        ax.bar(x + w/2, fid,  width=w, label="best_bell_fidelity")
        ax.set_xticks(x)
        ax.set_xticklabels(branches)
        ax.set_ylabel("Quality")
        ax.set_ylim(0, 1.05)
        ax.set_title("Conditional swapped-state quality")
        ax.legend()

        # (3) alpha-beta heatmap
        ax = axs[1, 0]
        im = ax.imshow(
            Z,
            origin="lower",
            aspect="auto",
            extent=[alphas.min(), alphas.max(), betas.min(), betas.max()],
        )
        ax.set_xlabel(UI_LABELS["alpha_A_deg"])
        ax.set_ylabel(UI_LABELS["alpha_B_deg"])
        ax.set_title(f"Heatmap: {metric} for {branch}")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        # (4) summary panel with inherited labels
        ax = axs[1, 1]
        ax.axis("off")

        extra_lines = [
            ("Focus branch", branch),
            ("Heatmap branch", heatmap_branch.value),
            ("Heatmap metric", metric),
            ("Grid", str(npts)),
            ("Link factor", f"{sim['summary']['link_factor']:.4f}"),
            ("Total conditional prob", f"{sim['summary']['total_conditional_prob']:.4f}"),
            ("Total valid prob", f"{sim['summary']['total_valid_prob']:.4f}"),
            ("Total entanglement yield", f"{sim['summary']['total_ent_yield']:.4f}"),
            ("Total fidelity yield", f"{sim['summary']['total_fid_yield']:.4f}"),
        ]

        lines = []
        lines.append("Current parameters")
        lines.append("------------------")
        lines.extend(build_parameter_lines(params))
        lines.append("")
        lines.append("Figure settings")
        lines.append("---------------")
        for label, value in extra_lines[:4]:
            lines.append(f"{label} = {value}")
        lines.append("")
        lines.append("Summary metrics")
        lines.append("---------------")
        for label, value in extra_lines[4:]:
            lines.append(f"{label} = {value}")

        txt = "\n".join(lines)

        ax.text(
            0.02, 0.98, txt,
            va="top",
            ha="left",
            family="monospace",
            fontsize=10.5,
        )

        plt.tight_layout()
        plt.show()

plot_btn.on_click(render_figures)

display(Markdown("## Figures"))
display(widgets.HBox([heatmap_branch, heatmap_metric, grid_pts, plot_btn]))
display(plot_out)

# initial render
render_figures()

## Figures

Output()